In [ ]:
import heapq

names = ["S", "A", "C", "D"]

INF = int(1e9)

def dijkstra(w, goal):
    n = len(w)
    dist = [INF] * n
    dist[goal] = 0
    pq = [(0, goal)]
    
    while pq:
        d, u = heapq.heappop(pq)
        if d != dist[u]:
            continue
        for v in range(n):
            if w[u][v] >= 0:
                nd = d + w[u][v]
                if nd < dist[v]:
                    dist[v] = nd
                    heapq.heappush(pq, (nd, v))
    return dist

def reconstruct_path(parent, start, goal):
    path = []
    cur = goal
    while cur != -1:
        path.append(cur)
        if cur == start:
            break
        cur = parent[cur]
    return path[::-1]

def run_a_star(w, h, start, goal):
    n = len(w)
    
    dist_to_goal = dijkstra(w, goal)
    
    # admissibility: heuristic <= true distances
    admissible = all(h[i] <= dist_to_goal[i] for i in range(n))
    
    # consistency: h(u) <= w(u,v) + h(v)
    consistent = True
    for u in range(n):
        for v in range(n):
            if w[u][v] >= 0 and h[u] > w[u][v] + h[v]:
                consistent = False
    
    g = [INF] * n
    parent = [-1] * n
    closed = [False] * n
    open_list = []
    
    g[start] = 0
    heapq.heappush(open_list, (g[start] + h[start], start))
    
    while open_list:
        f, u = heapq.heappop(open_list)
        if closed[u]:
            continue
        closed[u] = True
        if u == goal:
            break
        for v in range(n):
            if w[u][v] >= 0:
                cost = w[u][v]
                if g[u] + cost < g[v]:
                    g[v] = g[u] + cost
                    parent[v] = u
                    heapq.heappush(open_list, (g[v] + h[v], v))
    
    result = {}
    if g[goal] >= INF:
        result['cost'] = -1
        result['path'] = []
    else:
        result['cost'] = g[goal]
        result['path'] = reconstruct_path(parent, start, goal)
    
    result['dist_to_goal'] = dist_to_goal
    result['admissible'] = admissible
    result['consistent'] = consistent
    return result

def main():
    n = 4
    w = [[-1]*n for _ in range(n)]
    
    def add_edge(u, v, cost):
        w[u][v] = cost
        w[v][u] = cost
    
    add_edge(0, 1, 1)  
    add_edge(0, 2, 4)  
    add_edge(0, 3, 5)  
    add_edge(1, 2, 2)  
    add_edge(3, 2, 5)  
    
    print("Original graph edges and weights:")
    for i in range(n):
        for j in range(i+1, n):
            if w[i][j] >= 0:
                print(f"{names[i]} - {names[j]}: {w[i][j]}")
    print()
    
    h_exam = [7, 6, 2, 1]  
    print("Heuristic values (exam-provided):")
    for i in range(n):
        print(f"{names[i]}: {h_exam[i]}")
    print()
    
    start, goal = 0, 2  
    
    res_exam = run_a_star(w, h_exam, start, goal)
    print("A* result using exam heuristic:")
    if res_exam['cost'] < 0:
        print("No path found")
    else:
        print("Path cost:", res_exam['cost'])
        print("Path:", " -> ".join(names[i] for i in res_exam['path']))
    print("Heuristic admissible?", "Yes" if res_exam['admissible'] else "No")
    print("Heuristic consistent?", "Yes" if res_exam['consistent'] else "No")
    print()
    
    h_good = res_exam['dist_to_goal'] 
    print("Using admissible/consistent heuristic (true distances):")
    for i in range(n):
        print(f"{names[i]}: {h_good[i] if h_good[i]<INF else -1}")
    print()
    
    res_good = run_a_star(w, h_good, start, goal)
    print("A* result using admissible/consistent heuristic:")
    if res_good['cost'] < 0:
        print("No path found")
    else:
        print("Path cost:", res_good['cost'])
        print("Path:", " -> ".join(names[i] for i in res_good['path']))
    print("Heuristic admissible?", "Yes" if res_good['admissible'] else "No")
    print("Heuristic consistent?", "Yes" if res_good['consistent'] else "No")
    print()
    
    print("Comparison:")
    print(f"Exam heuristic path cost {res_exam['cost']}, admissible/consistent heuristic path cost {res_good['cost']}")
    print("Comment: When the heuristic is admissible and consistent, A* guarantees optimal path. The exam heuristic may overestimate or be inconsistent, possibly giving suboptimal path.")

if __name__ == "__main__":
    main()


Original graph edges and weights:
S - A: 1
S - C: 4
S - D: 5
A - C: 2
C - D: 5

Heuristic values (exam-provided):
S: 7
A: 6
C: 2
D: 1

A* result using exam heuristic:
Path cost: 4
Path: S -> C
Heuristic admissible? No
Heuristic consistent? No

Using admissible/consistent heuristic (true distances):
S: 3
A: 2
C: 0
D: 5

A* result using admissible/consistent heuristic:
Path cost: 3
Path: S -> A -> C
Heuristic admissible? Yes
Heuristic consistent? Yes

Comparison:
Exam heuristic path cost 4, admissible/consistent heuristic path cost 3
Comment: When the heuristic is admissible and consistent, A* guarantees optimal path. The exam heuristic may overestimate or be inconsistent, possibly giving suboptimal path.
